# DM Analysis Notebook (Spark)
This notebook analyses sample election results with explanatory markdown between each code cell, using PySpark for parallel file loading and computation.

In [ ]:
from pathlib import Path  # Work with file-system paths safely.
from pyspark.sql import SparkSession  # Create and configure a Spark session.
from pyspark.sql import functions as F  # Use Spark SQL functions for transformations.
from pyspark.sql.window import Window  # Define window functions for winner selection.

spark_builder = SparkSession.Builder()

spark = (
    spark_builder
    .appName("DM Analysis Spark")
    .master("spark://localhost:7077")  # Connect to Docker Spark master.
    .config("spark.executor.instances", "3")  # Target three worker executors.
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")  # Keep notebook output focused on results.
print(f"Spark master: {spark.sparkContext.master}")

Spark master: spark://localhost:7077


## Step 1: Locate Input Files
The next code cell discovers all sample result JSON files.

In [5]:
base_path = Path("resources/sample-election-results")  # Point to the local sample results directory.
spark_base_path = "file:///tmp/election-data"  # Shared absolute path copied to host and Spark workers.
json_files = sorted(base_path.glob("*.json"))  # Collect and sort every JSON result file.
print(f"Files found: {len(json_files)}")  # Show how many files were discovered.
print(json_files[:5])  # Preview the first few paths for validation.

Files found: 650
[PosixPath('resources/sample-election-results/result001.json'), PosixPath('resources/sample-election-results/result002.json'), PosixPath('resources/sample-election-results/result003.json'), PosixPath('resources/sample-election-results/result004.json'), PosixPath('resources/sample-election-results/result005.json')]


## Step 2: Load and Validate Records in Parallel
The next cell loads all files at once through Spark's distributed JSON reader so parsing runs in parallel.

In [6]:
results_df = (
    spark.read
    .option("multiLine", True)
    .json(f"{spark_base_path}/*.json")  # Read all JSON files in parallel from worker-visible path.
)

results_df = results_df.cache()  # Cache because the DataFrame is reused in later steps.
print(f"Records loaded: {results_df.count()}")  # Trigger read and confirm row count.
results_df.printSchema()  # Preview the inferred schema.

ERROR:root:KeyboardInterrupt while sending command.               (0 + 0) / 650]
Traceback (most recent call last):
  File "/Library/Desenvolvimento/uk/bbc/software-engineering-technical-assessments/election-api-python/.venv/lib/python3.13/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Library/Desenvolvimento/uk/bbc/software-engineering-technical-assessments/election-api-python/.venv/lib/python3.13/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 

## Step 3: Compute Basic Metrics
This cell computes constituency counts and total-vote statistics from party results using Spark aggregations.

In [7]:
constituency_key = F.coalesce(
    F.col("constituencyId").cast("string"),
    F.col("constituencyName").cast("string"),
    F.input_file_name()
)

constituency_totals = (
    results_df
    .select(constituency_key.alias("constituency_key"), F.explode_outer("partyResults").alias("party_result"))
    .groupBy("constituency_key")
    .agg(F.sum(F.coalesce(F.col("party_result.votes"), F.lit(0))).alias("total_votes"))
)

metrics = constituency_totals.agg(
    F.count("*").alias("total_constituencies"),
    F.avg("total_votes").alias("average_total_votes"),
    F.max("total_votes").alias("max_total_votes"),
    F.min("total_votes").alias("min_total_votes")
).collect()[0]

print(f"Constituencies: {metrics['total_constituencies']}")
print(f"Average total votes: {metrics['average_total_votes']:.2f}")
print(f"Max total votes: {metrics['max_total_votes']:.2f}")
print(f"Min total votes: {metrics['min_total_votes']:.2f}")

NameError: name 'results_df' is not defined

## Step 4: Quick Party Frequency Scan
The final code cell derives each winner from party vote totals and counts party wins using window functions.

In [ ]:
party_votes = (
    results_df
    .select(constituency_key.alias("constituency_key"), F.explode_outer("partyResults").alias("party_result"))
    .select(
        "constituency_key",
        F.coalesce(F.col("party_result.party"), F.lit("UNKNOWN")).alias("party"),
        F.coalesce(F.col("party_result.votes"), F.lit(0)).alias("votes")
    )
)

winner_window = Window.partitionBy("constituency_key").orderBy(F.col("votes").desc(), F.col("party").asc())
winners = (
    party_votes
    .withColumn("rank", F.row_number().over(winner_window))
    .filter(F.col("rank") == 1)
    .select("party")
)

party_counts = winners.groupBy("party").count().orderBy(F.col("count").desc(), F.col("party").asc())
print("Top party counts:")
for row in party_counts.limit(10).collect():
    print(f"{row['party']}: {row['count']}")

## Optional Cleanup
Stop Spark when you are done to release local resources.

In [ ]:
# spark.stop()